
# Moscow Macro Parser — quarterly macro dataset for the market block

Этот ноутбук собирает **квартальные макроэкономические ряды для Москвы** в один `CSV`, который потом можно использовать как вход для `market block`.

## Что попадает в итоговый CSV

- **Ключевая ставка ЦБ**: средняя за квартал, значение на конец квартала, изменение к/к.
- **Курсы ЦБ**: USD/RUB и EUR/RUB — средние за квартал, конец квартала, лог-доходности.
- **Ипотека по Москве (ЦБ, региональный разрез)**:
  - общее число ипотечных кредитов,
  - общий объём ипотечных кредитов,
  - средневзвешенная ставка,
  - аналогичные ряды по ипотеке под **ДДУ**.
- **Индекс московской недвижимости Домклик / MOEX (`MREDC`)**.
- **Дополнительные биржевые индексы MOEX** (по желанию): `IMOEX`, `RGBI`.
- **Официальные индексы и статистика Мосстата**:
  - индекс цен на первичном рынке жилья,
  - ИПЦ по Москве,
  - ввод жилья в Москве.
- **Опциональные policy-флаги** по льготной ипотеке.

## Идея

В market block лучше не передавать всю эту сырую макроинформацию напрямую в сделочную hedonic-модель.  
Обычно удобнее сначала собрать и смоделировать отдельный квартальный `market_state`, а уже потом отдавать в hedonic один-два агрегированных признака (`market_log_price`, `market_return`).

Но сначала нужен **чистый и воспроизводимый квартальный макро-датасет** — именно его и строит этот ноутбук.


## Источники

Официальные источники, под которые написаны парсеры:

- Банк России — ключевая ставка: `https://www.cbr.ru/hd_base/keyrate/`
- Банк России — официальные курсы валют: `https://www.cbr.ru/scripts/XML_daily_dyn.asp`
- Банк России — ипотечная статистика по регионам и по ДДУ (modern monthly tables): `https://www.cbr.ru/statistics/bank_sector/mortgage/`
- Банк России — ретроспективный архив первичного ипотечного рынка (`retro2`, таблица `4-6`, Excel bundle со всеми месяцами до 01.01.2019): `https://www.cbr.ru/statistics/bank_sector/mortgage/archiv/retro2/`
- Банк России — текущие бюллетени `mortgage_lending_market_*.xlsx` / `*.pdf` как дополнительный discovery-layer для проверки доступных публикаций
- MOEX ISS — история индексов: `https://iss.moex.com/iss/history/engines/stock/markets/index/securities/{SECID}.json`
- MOEX — карточка индекса `MREDC`: `https://www.moex.com/ru/index/MREDC`
- Мосстат — цены и тарифы / индексы цен на первичном рынке жилья
- Мосстат — ИПЦ по Москве
- Мосстат — ввод жилья в Москве

Парсеры сделаны так, чтобы:

- сначала брать актуальные modern monthly таблицы ЦБ,
- затем автоматически подмешивать ретро-историю по Москве из `retro2` как backfill,
- а PDF fallback использовать только если после XLSX-backfill остаются ранние дыры по нужным показателям.


In [ ]:
from __future__ import annotations

import io
import json
import math
import re
import warnings
import xml.etree.ElementTree as ET
from dataclasses import dataclass
from datetime import date, datetime
from pathlib import Path
from typing import Iterable, Optional
from urllib.parse import urljoin, urlencode

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
pd.options.display.max_columns = 200
pd.options.display.width = 240


In [ ]:
# ── Конфигурация ─────────────────────────────────────────────────────────────
PROJECT_DIR = Path("./cashflow_project")
OUT_DIR = PROJECT_DIR / "data" / "macro"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Берём максимально длинную историю для макро-блока.
# Ключевая ставка доступна с 17.09.2013, валютные курсы и часть ипотечных рядов — раньше.
START_DATE = "2013-01-01"
END_DATE   = None  # None => today
REGION_NAME = "г. Москва"
REGION_SHORT = "Moscow"

OUTPUT_CSV = OUT_DIR / f"macro_quarterly_{REGION_SHORT}.csv"
OUTPUT_COVERAGE_CSV = OUT_DIR / f"macro_quarterly_{REGION_SHORT}_coverage.csv"
OUTPUT_MORTGAGE_MONTHLY_CSV = OUT_DIR / f"mortgage_monthly_{REGION_SHORT}.csv"

# Для XML-курсов ЦБ
CBR_VAL_IDS = {
    "USD": "R01235",
    "EUR": "R01239",
}

# Полная история ключевой ставки через query-параметры страницы ЦБ.
# Именно так мы избегаем дефолтной "короткой витрины" за последние дни.
KEY_RATE_HISTORY_URL = "https://www.cbr.ru/hd_base/KeyRate/"

# Ипотека ЦБ: modern monthly региональные ряды.
# Их приоритет выше любого ретро-источника при сшивке истории.
CBR_MORTGAGE_URLS = {
    "mortgage_total_count": {
        "url": "https://www.cbr.ru/vfs/statistics/BankSector/Mortgage/02_10_Quantity_mortgage.xlsx",
        "sheet": "в рублях",
        "value_name": "mortgage_total_count_monthly",
    },
    "mortgage_total_volume": {
        "url": "https://www.cbr.ru/vfs/statistics/BankSector/Mortgage/02_11_New_loans_mortgage.xlsx",
        "sheet": "в рублях",
        "value_name": "mortgage_total_volume_mln_rub_monthly",
    },
    "mortgage_total_rate": {
        "url": "https://www.cbr.ru/vfs/statistics/BankSector/Mortgage/02_13_Rates_mortgage.xlsx",
        "sheet": "ставка в рублях",
        "value_name": "mortgage_total_rate_pct_monthly",
    },
    "mortgage_ddu_count": {
        "url": "https://www.cbr.ru/vfs/statistics/BankSector/Mortgage/02_15_Quantity_scpa_mortgage.xlsx",
        "sheet": "в рублях",
        "value_name": "mortgage_ddu_count_monthly",
    },
    "mortgage_ddu_volume": {
        "url": "https://www.cbr.ru/vfs/statistics/BankSector/Mortgage/02_16_New_loans_scpa_mortgage.xlsx",
        "sheet": "в рублях",
        "value_name": "mortgage_ddu_volume_mln_rub_monthly",
    },
    "mortgage_ddu_rate": {
        "url": "https://www.cbr.ru/vfs/statistics/BankSector/Mortgage/02_17_Rates_scpa_mortgage.xlsx",
        "sheet": "ставка в рублях",
        "value_name": "mortgage_ddu_rate_pct_monthly",
    },
}

# MOEX ISS
MOEX_ISS_HISTORY_URL = "https://iss.moex.com/iss/history/engines/stock/markets/index/securities/{secid}.json"
MOEX_INDEX_SECIDS = ["MREDC", "IMOEX", "RGBI"]

# Мосстат: страницы-разделы, с которых ищем актуальные XLSX
MOSSTAT_PAGES = {
    "primary_price_index": "https://77.rosstat.gov.ru/folder/64508",
    "cpi": "https://77.rosstat.gov.ru/folder/64640",
    "housing_completion": "https://77.rosstat.gov.ru/folder/64519",
}

# Поисковые маски по href / тексту ссылки
MOSSTAT_PATTERNS = {
    "primary_price_index": [r"первичн.*рынк.*жиль", r"индекс.*цен"],
    "cpi": [r"индекс.*потребительск", r"товар.*услуг"],
    "housing_completion": [r"ввод.*жил.*дом"],
}

REQUEST_TIMEOUT = 60
SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "cashflow-macro-parser/1.2 (research notebook)"
})

print("OUTPUT_CSV:", OUTPUT_CSV)
print("OUTPUT_MORTGAGE_MONTHLY_CSV:", OUTPUT_MORTGAGE_MONTHLY_CSV)


## Вспомогательные функции

In [ ]:
RU_MONTHS = {
    "январь": 1, "февраль": 2, "март": 3, "апрель": 4,
    "май": 5, "июнь": 6, "июль": 7, "август": 8,
    "сентябрь": 9, "октябрь": 10, "ноябрь": 11, "декабрь": 12,
}


def parse_ru_month_year(text: object) -> pd.Timestamp:
    if pd.isna(text):
        return pd.NaT
    s = str(text).strip().lower().replace("ё", "е").replace("\xa0", " ").replace(" ", " ")
    m = re.match(r"(.+?)\s+(\d{4})$", s)
    if not m:
        return pd.NaT
    month = RU_MONTHS.get(m.group(1).strip())
    year = int(m.group(2))
    if month is None:
        return pd.NaT
    return pd.Timestamp(year=year, month=month, day=1)


def to_numeric_safe(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.replace("\xa0", " ", regex=False).str.replace(" ", " ", regex=False).str.strip()
    s = s.str.replace("%", "", regex=False)
    s = s.str.replace(",", ".", regex=False)
    s = s.str.replace(r"[^0-9.\-]", "", regex=True)
    s = s.replace({"": np.nan, "nan": np.nan, "None": np.nan})
    out = pd.to_numeric(s, errors="coerce")

    # На отдельных витринах ЦБ ставка иногда приезжает как 1500 вместо 15.00.
    # Сохраняем исходную логику ноутбука, чтобы не ломать уже существующие downstream CSV.
    out = out.where(~(out > 100), out / 100.0)
    return out


def clip_date_range(df: pd.DataFrame, date_col: str = "date") -> pd.DataFrame:
    out = df.copy()
    out[date_col] = pd.to_datetime(out[date_col])
    start_ts = pd.Timestamp(START_DATE)
    end_ts = pd.Timestamp.today().normalize() if END_DATE is None else pd.Timestamp(END_DATE)
    return out[(out[date_col] >= start_ts) & (out[date_col] <= end_ts)].copy()


def add_quarter_columns(df: pd.DataFrame, date_col: str = "date") -> pd.DataFrame:
    out = df.copy()
    out[date_col] = pd.to_datetime(out[date_col])
    out["quarter"] = out[date_col].dt.to_period("Q").astype(str)
    out["quarter_start"] = out[date_col].dt.to_period("Q").dt.start_time
    out["quarter_end"] = out[date_col].dt.to_period("Q").dt.end_time
    return out


def aggregate_daily_to_quarter(df: pd.DataFrame, value_col: str, prefix: str) -> pd.DataFrame:
    tmp = add_quarter_columns(df, "date")
    q = (
        tmp.groupby("quarter", as_index=False)
        .agg(
            **{
                f"{prefix}_avg_q": (value_col, "mean"),
                f"{prefix}_eoq": (value_col, lambda s: s.dropna().iloc[-1] if s.dropna().shape[0] else np.nan),
                f"{prefix}_min_q": (value_col, "min"),
                f"{prefix}_max_q": (value_col, "max"),
                f"{prefix}_obs_n": (value_col, "size"),
            }
        )
        .sort_values("quarter")
        .reset_index(drop=True)
    )
    if f"{prefix}_eoq" in q.columns:
        prev = q[f"{prefix}_eoq"].shift(1)
        q[f"{prefix}_log_return_q"] = np.where(
            (q[f"{prefix}_eoq"] > 0) & (prev > 0),
            np.log(q[f"{prefix}_eoq"]) - np.log(prev),
            np.nan,
        )
        q[f"{prefix}_qoq_change"] = q[f"{prefix}_eoq"] - prev
    return q


def aggregate_monthly_to_quarter(df: pd.DataFrame, value_col: str, prefix: str, how: str = "sum") -> pd.DataFrame:
    tmp = add_quarter_columns(df, "date")
    if how == "sum":
        val = (value_col, lambda s: s.sum(min_count=1))
    elif how == "mean":
        val = (value_col, "mean")
    elif how == "last":
        val = (value_col, lambda s: s.dropna().iloc[-1] if s.dropna().shape[0] else np.nan)
    else:
        raise ValueError(f"Unsupported how={how}")

    q = (
        tmp.groupby("quarter", as_index=False)
        .agg(**{f"{prefix}": val})
        .sort_values("quarter")
        .reset_index(drop=True)
    )
    prev = q[f"{prefix}"].shift(1)
    q[f"{prefix}_qoq_change"] = q[f"{prefix}"] - prev
    q[f"{prefix}_log_return_q"] = np.where(
        (q[f"{prefix}"] > 0) & (prev > 0),
        np.log(q[f"{prefix}"]) - np.log(prev),
        np.nan,
    )
    return q


def make_quarter_spine(start: str = START_DATE, end: Optional[str] = END_DATE) -> pd.DataFrame:
    start_ts = pd.Timestamp(start).to_period("Q").start_time
    end_ts = (pd.Timestamp.today() if end is None else pd.Timestamp(end)).to_period("Q").start_time
    qs = pd.period_range(start=start_ts, end=end_ts, freq="Q")
    return pd.DataFrame({"quarter": qs.astype(str)})


def outer_merge_on_quarter(frames: list[pd.DataFrame]) -> pd.DataFrame:
    out = make_quarter_spine()
    for df in frames:
        if df is None or df.empty:
            continue
        cols = [c for c in df.columns if c == "quarter" or c not in out.columns]
        out = out.merge(df[cols], on="quarter", how="left")
    return out


def build_coverage_table(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for col in df.columns:
        if col == "quarter":
            continue
        s = df[["quarter", col]].dropna()
        rows.append({
            "feature": col,
            "first_quarter": s["quarter"].iloc[0] if len(s) else None,
            "last_quarter": s["quarter"].iloc[-1] if len(s) else None,
            "non_null_quarters": int(len(s)),
        })
    return pd.DataFrame(rows).sort_values(["first_quarter", "feature"], na_position="last").reset_index(drop=True)


## 1. Ключевая ставка ЦБ

In [ ]:
def fetch_key_rate_history(start: str = START_DATE, end: str | None = END_DATE) -> pd.DataFrame:
    """
    Полная история ключевой ставки ЦБ.

    Почему предыдущая версия падала:
    - страница ЦБ часто отдаёт не обычную HTML-таблицу, а разметку, где pandas.read_html
      не всегда стабильно распознаёт заголовки;
    - иногда query-параметры диапазона игнорируются;
    - в результате таблицы находились, но нужные колонки не распознавались,
      поэтому last_err оставался None.

    Эта версия:
    1) явно запрашивает страницу с диапазоном дат;
    2) сначала пытается распарсить таблицу через read_html;
    3) если не получилось — парсит строки вида DD.MM.YYYY RATE регуляркой прямо из HTML-текста;
    4) если и это не удалось — делает fallback на список дат изменения ставки из XLSX,
       а затем разворачивает его в daily step function.
    """
    import re

    start_ts = pd.Timestamp(start).normalize()
    end_ts = pd.Timestamp.today().normalize() if end is None else pd.Timestamp(end).normalize()

    params = {
        'UniDbQuery.Posted': 'True',
        'UniDbQuery.From': start_ts.strftime('%d.%m.%Y'),
        'UniDbQuery.To': end_ts.strftime('%d.%m.%Y'),
    }
    headers = {
        'User-Agent': 'Mozilla/5.0',
        'Accept-Language': 'ru,en;q=0.9',
    }

    def _finalize_changes(df_changes: pd.DataFrame) -> pd.DataFrame:
        out = df_changes.copy()
        out['date'] = pd.to_datetime(out['date'], dayfirst=True, errors='coerce')
        out['key_rate_pct'] = to_numeric_safe(out['key_rate_pct'])
        out = out.dropna(subset=['date', 'key_rate_pct']).sort_values('date').drop_duplicates('date')
        out = out[out['date'] >= start_ts]
        if out.empty:
            raise RuntimeError('История ключевой ставки получена пустой после очистки.')

        daily_index = pd.date_range(out['date'].min(), end_ts, freq='D')
        stepped = pd.DataFrame({'date': daily_index})
        stepped = stepped.merge(out, on='date', how='left')
        stepped['key_rate_pct'] = stepped['key_rate_pct'].ffill()
        stepped = stepped[stepped['date'] >= start_ts].reset_index(drop=True)
        return stepped

    # --- 1) Основной путь: HTML-страница ключевой ставки ---
    for url in [KEY_RATE_HISTORY_URL, 'https://www.cbr.ru/eng/hd_base/KeyRate/']:
        try:
            resp = SESSION.get(url, params=params, headers=headers, timeout=REQUEST_TIMEOUT)
            resp.raise_for_status()
            html = resp.text

            # 1a. пробуем read_html
            try:
                tables = pd.read_html(io.StringIO(html))
            except Exception:
                tables = []

            for df in tables:
                cur = df.copy()
                if isinstance(cur.columns, pd.MultiIndex):
                    cur.columns = [' '.join([str(x) for x in tup if str(x) != 'nan']).strip() for tup in cur.columns]
                cur.columns = [str(c).strip().lower() for c in cur.columns]

                date_col = next((c for c in cur.columns if 'дат' in c or c == 'date'), None)
                rate_col = next((c for c in cur.columns if 'став' in c or c == 'rate'), None)
                if date_col and rate_col:
                    parsed = cur[[date_col, rate_col]].rename(columns={date_col: 'date', rate_col: 'key_rate_pct'})
                    parsed = parsed.dropna(how='all')
                    if len(parsed) >= 3:
                        return _finalize_changes(parsed)

            # 1b. fallback: regex по тексту HTML
            # Ищем строки вида 03.04.2026 15,00 / 03.04.2026 15.00
            pairs = re.findall(r'(\d{2}\.\d{2}\.\d{4})\s+([0-9]+(?:[\.,][0-9]+)?)', html)
            if pairs:
                parsed = pd.DataFrame(pairs, columns=['date', 'key_rate_pct'])
                parsed = parsed.drop_duplicates()
                if parsed['date'].nunique() >= 10:
                    return _finalize_changes(parsed)
        except Exception:
            pass

    # --- 2) Fallback: XLSX с датами изменения ставки ---
    # Официальный файл ЦБ с датами изменения ставки денежно-кредитной политики.
    xlsx_urls = [
        'https://www.cbr.ru/vfs/hd_base/procstav/ir_chg_mpo/ir_chg_mpo.xlsx',
        'https://www.cbr.ru/vfs/hd_base/procstav/ir_chg_mpo/ir_chg_mpo_e.xlsx',
    ]
    for xurl in xlsx_urls:
        try:
            chg = pd.read_excel(xurl)
            chg.columns = [str(c).strip().lower() for c in chg.columns]
            date_col = next((c for c in chg.columns if 'date' in c or 'дат' in c), None)
            rate_col = next((c for c in chg.columns if 'key rate' in c or 'ключевая ставка' in c or c == 'key rate'), None)
            if date_col and rate_col:
                parsed = chg[[date_col, rate_col]].rename(columns={date_col: 'date', rate_col: 'key_rate_pct'})
                parsed = parsed.dropna(how='all')
                if len(parsed) >= 3:
                    return _finalize_changes(parsed)
        except Exception:
            pass

    raise RuntimeError(
        'Не удалось получить историю ключевой ставки ни со страницы ЦБ, ни из fallback XLSX. '
        'Проверь доступ к cbr.ru из среды запуска или обнови парсер под текущую разметку страницы.'
    )


key_rate_daily = fetch_key_rate_history()
key_rate_q = aggregate_daily_to_quarter(key_rate_daily, 'key_rate_pct', 'key_rate')
display(key_rate_q.head())
display(key_rate_q.tail())



## 2. Официальные курсы ЦБ: USD/RUB и EUR/RUB

In [ ]:

def fetch_cbr_fx_daily(char_code: str, val_id: str, start: str = START_DATE, end: Optional[str] = END_DATE) -> pd.DataFrame:
    end_eff = pd.Timestamp.today().strftime("%d/%m/%Y") if end is None else pd.Timestamp(end).strftime("%d/%m/%Y")
    start_eff = pd.Timestamp(start).strftime("%d/%m/%Y")
    url = (
        "https://www.cbr.ru/scripts/XML_dynamic.asp?" +
        urlencode({"date_req1": start_eff, "date_req2": end_eff, "VAL_NM_RQ": val_id})
    )
    r = SESSION.get(url, timeout=REQUEST_TIMEOUT)
    r.raise_for_status()

    root = ET.fromstring(r.content)
    rows = []
    for rec in root.findall("Record"):
        d = rec.attrib.get("Date")
        value = rec.findtext("Value")
        nominal = rec.findtext("Nominal")
        rows.append({
            "date": pd.to_datetime(d, dayfirst=True, errors="coerce"),
            f"{char_code.lower()}_rub": pd.to_numeric(str(value).replace(",", "."), errors="coerce"),
            f"{char_code.lower()}_nominal": pd.to_numeric(str(nominal).replace(",", "."), errors="coerce"),
        })

    df = pd.DataFrame(rows).dropna(subset=["date"]).sort_values("date")
    val_col = f"{char_code.lower()}_rub"
    nom_col = f"{char_code.lower()}_nominal"
    df[val_col] = df[val_col] / df[nom_col].replace(0, np.nan)
    df = df[["date", val_col]].copy()
    return clip_date_range(df)


usd_daily = fetch_cbr_fx_daily("USD", CBR_VAL_IDS["USD"])
eur_daily = fetch_cbr_fx_daily("EUR", CBR_VAL_IDS["EUR"])

usd_q = aggregate_daily_to_quarter(usd_daily, "usd_rub", "usd_rub")
eur_q = aggregate_daily_to_quarter(eur_daily, "eur_rub", "eur_rub")

display(usd_q.tail())


## 3. Ипотека по Москве из региональной статистики ЦБ

## 3a. Важное замечание по покрытию рядов и retro-backfill

У разных источников разная глубина истории.

В этой версии ноутбука ипотечный блок строится в 3 слоя:

1. `modern_monthly`
   - актуальные regional monthly XLSX Банка России,
   - они имеют высший приоритет при совпадении дат.

2. `retro_xlsx`
   - ретроспективный архив ЦБ `retro2`, таблица `4-6`,
   - из него восстанавливаются ранние месяцы по Москве до `2019-01-01`.

3. `retro_pdf_fallback`
   - используется только если после XLSX-backfill остаются ранние дыры,
   - и только с явным логированием причин, если парсинг PDF в среде недоступен.

Важно:

- если один и тот же месяц есть и в modern, и в retro, побеждает modern;
- если показатель реально отсутствует в раннем источнике, он остаётся `NaN`, а не превращается в ноль;
- `MREDC` отдельно назад не расширяем: у него собственный официальный старт.


In [ ]:
from mortgage_backfill import (
    build_mortgage_monthly_current,
    build_mortgage_monthly_backfilled,
    build_mortgage_quarterly_features,
    compare_coverage,
)

mortgage_monthly_current, mortgage_current_diag = build_mortgage_monthly_current(
    cbr_mortgage_urls=CBR_MORTGAGE_URLS,
    region_name=REGION_NAME,
    start_date=START_DATE,
    end_date=END_DATE,
    session=SESSION,
    logger=print,
    numeric_parser=to_numeric_safe,
)

mortgage_monthly_backfilled, mortgage_backfill_diag = build_mortgage_monthly_backfilled(
    cbr_mortgage_urls=CBR_MORTGAGE_URLS,
    region_name=REGION_NAME,
    start_date=START_DATE,
    end_date=END_DATE,
    session=SESSION,
    logger=print,
    numeric_parser=to_numeric_safe,
)

mortgage_monthly_backfilled.to_csv(OUTPUT_MORTGAGE_MONTHLY_CSV, index=False, encoding="utf-8-sig")
print("Saved mortgage monthly:", OUTPUT_MORTGAGE_MONTHLY_CSV)

retro_sources = mortgage_backfill_diag.get("sources", {})
print("Retro Excel source:", retro_sources.get("retro_excel_url"))
print("Retro PDF fallback candidates:", retro_sources.get("retro_pdf_candidates", []))
if retro_sources.get("stat_morgage_tables"):
    print("Discovered stat_morgage_tables URLs:", retro_sources["stat_morgage_tables"])
else:
    print("stat_morgage_tables probe: no stable public URLs discovered, using retro2 Excel bundle instead.")
if retro_sources.get("current_bulletin_xlsx"):
    print("Current bulletin XLSX sample:", retro_sources["current_bulletin_xlsx"][:2])

print("Modern mortgage monthly coverage:", mortgage_current_diag.get("first_date"), "->", mortgage_current_diag.get("last_date"), "rows=", mortgage_current_diag.get("rows"))
print("Backfilled mortgage monthly coverage:", mortgage_backfill_diag.get("combined_first_date"), "->", mortgage_backfill_diag.get("combined_last_date"), "rows=", mortgage_backfill_diag.get("combined_rows"))
print("Remaining pre-start gaps after retro XLSX:", mortgage_backfill_diag.get("gap_metrics_after_retro_xlsx", []))

display(mortgage_monthly_backfilled.head(4))
display(mortgage_monthly_backfilled.tail(4))


In [ ]:
mortgage_q_current = build_mortgage_quarterly_features(mortgage_monthly_current)
mortgage_q = build_mortgage_quarterly_features(mortgage_monthly_backfilled)

mortgage_feature_list = sorted(
    set([c for c in mortgage_q_current.columns if c.startswith("mortgage_")])
    | set([c for c in mortgage_q.columns if c.startswith("mortgage_")])
)

mortgage_coverage_before = build_coverage_table(mortgage_q_current)
mortgage_coverage_after = build_coverage_table(mortgage_q)
mortgage_coverage_compare = compare_coverage(
    mortgage_coverage_before,
    mortgage_coverage_after,
    features=mortgage_feature_list,
)

print("Mortgage quarterly coverage before backfill:")
display(mortgage_coverage_before)

print("Mortgage quarterly coverage after backfill:")
display(mortgage_coverage_after)

print("Mortgage coverage delta:")
display(
    mortgage_coverage_compare[
        [
            "feature",
            "first_quarter_before",
            "first_quarter_after",
            "last_quarter_before",
            "last_quarter_after",
            "non_null_quarters_before",
            "non_null_quarters_after",
            "history_extended_backward",
        ]
    ]
)

print("Backfilled mortgage quarterly head:")
display(mortgage_q.head(12))

print("Backfilled mortgage quarterly tail:")
display(mortgage_q.tail())


## 4. MOEX ISS: индекс московской недвижимости Домклик (MREDC) и другие индексы

In [ ]:

def fetch_moex_index_history(secid: str, start: str = START_DATE, end: Optional[str] = END_DATE) -> pd.DataFrame:
    end_eff = pd.Timestamp.today().strftime("%Y-%m-%d") if end is None else pd.Timestamp(end).strftime("%Y-%m-%d")
    start_eff = pd.Timestamp(start).strftime("%Y-%m-%d")

    rows = []
    start_offset = 0
    page_size = 100

    while True:
        url = MOEX_ISS_HISTORY_URL.format(secid=secid)
        params = {
            "from": start_eff,
            "till": end_eff,
            "start": start_offset,
        }
        r = SESSION.get(url, params=params, timeout=REQUEST_TIMEOUT)
        r.raise_for_status()
        payload = r.json()

        hist = payload.get("history", {})
        cols = hist.get("columns", [])
        data = hist.get("data", [])
        if not data:
            break
        batch = pd.DataFrame(data, columns=cols)
        rows.append(batch)

        cursor = payload.get("history.cursor", {})
        cursor_data = cursor.get("data", [])
        if not cursor_data:
            break
        total = cursor_data[0][1]
        page_size = cursor_data[0][2]
        start_offset += page_size
        if start_offset >= total:
            break

    if not rows:
        return pd.DataFrame(columns=["date", f"{secid.lower()}_close"])

    df = pd.concat(rows, ignore_index=True)
    df["date"] = pd.to_datetime(df["TRADEDATE"], errors="coerce")
    out = df[["date", "CLOSE"]].rename(columns={"CLOSE": f"{secid.lower()}_close"}).copy()
    out = out.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)
    return clip_date_range(out)


moex_index_q_frames = []
for secid in MOEX_INDEX_SECIDS:
    daily = fetch_moex_index_history(secid)
    q = aggregate_daily_to_quarter(daily, f"{secid.lower()}_close", f"{secid.lower()}")
    moex_index_q_frames.append(q)
    print(secid, daily.shape)
    display(q.tail(2))


## 5. Мосстат: поиск актуальных XLSX по странице и парсинг в кварталы

In [ ]:

def find_first_matching_xlsx(page_url: str, include_patterns: list[str]) -> str:
    r = SESSION.get(page_url, timeout=REQUEST_TIMEOUT)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")

    candidates = []
    for a in soup.find_all("a", href=True):
        href = a.get("href")
        text = a.get_text(" ", strip=True)
        blob = f"{text} {href}".lower()
        if ".xlsx" not in href.lower() and ".xls" not in href.lower():
            continue
        if all(re.search(pat, blob, flags=re.I) for pat in include_patterns):
            candidates.append(urljoin(page_url, href))

    if not candidates:
        raise RuntimeError(f"Не нашли XLS/XLSX на странице {page_url} по паттернам {include_patterns}")
    return candidates[0]


def _guess_year_quarter_pairs(raw: pd.DataFrame) -> pd.DataFrame:
    """
    Пробуем найти матрицу, где годы и кварталы лежат в таблице. Функция не идеальна,
    но обычно хорошо работает для компактных xlsx Мосстата.
    """
    txt = raw.copy().astype(str)
    txt = txt.replace("nan", np.nan)

    # Ищем годы в верхних строках
    year_positions = []
    for i in range(min(8, raw.shape[0])):
        for j in range(raw.shape[1]):
            cell = str(raw.iat[i, j]).strip()
            if re.fullmatch(r"20\d{2}", cell):
                year_positions.append((i, j, int(cell)))

    # Ищем квартальные подписи
    quarter_rows = []
    for i in range(raw.shape[0]):
        row_blob = " | ".join(map(str, raw.iloc[i].tolist())).lower()
        if "кварт" in row_blob:
            quarter_rows.append(i)

    pairs = []
    if year_positions and quarter_rows:
        for i in quarter_rows:
            row = raw.iloc[i]
            for j, val in enumerate(row):
                qtxt = str(val).lower()
                qm = re.search(r"([ivx]+)\s*кварт", qtxt)
                if qm:
                    roman = qm.group(1).upper()
                    qmap = {"I":1, "II":2, "III":3, "IV":4}
                    qnum = qmap.get(roman)
                    if qnum is None:
                        continue
                    # Ищем ближайший год сверху в этой же колонке
                    col_year = None
                    for yi, yj, yy in year_positions:
                        if yj == j and yi < i:
                            col_year = yy
                    if col_year is not None:
                        pairs.append({"quarter": f"{col_year}Q{qnum}", "row_idx": i, "col_idx": j})
    return pd.DataFrame(pairs)


def parse_mosstat_primary_price_index(url: str) -> pd.DataFrame:
    xls = pd.ExcelFile(url)
    rows = []
    for sheet in xls.sheet_names:
        raw = pd.read_excel(url, sheet_name=sheet, header=None)
        # Ищем строку "Все типы квартир"
        target_rows = raw.index[
            raw.apply(lambda r: r.astype(str).str.contains("Все типы квартир", case=False, na=False).any(), axis=1)
        ].tolist()
        if not target_rows:
            continue

        pairs = _guess_year_quarter_pairs(raw)
        if pairs.empty:
            continue

        # Сценарий 1: кварталы расположены по колонкам, target row содержит сами значения
        target_idx = target_rows[0]
        for _, rec in pairs.iterrows():
            j = int(rec["col_idx"])
            value = raw.iat[target_idx, j]
            rows.append({"quarter": rec["quarter"], "mosstat_primary_price_index_q": pd.to_numeric(value, errors="coerce")})

    out = pd.DataFrame(rows).dropna().drop_duplicates(subset=["quarter"]).sort_values("quarter")
    if out.empty:
        raise RuntimeError("Не удалось распарсить индекс цен первичного жилья из xlsx Мосстата")
    out["mosstat_primary_price_index_q_qoq_change"] = out["mosstat_primary_price_index_q"].diff()
    return out


def parse_mosstat_cpi(url: str) -> pd.DataFrame:
    raw = pd.read_excel(url, header=None)
    years = [int(x) for x in pd.Series(raw.iloc[3].tolist()).astype(str).str.extract(r"(20\d{2})", expand=False).dropna().unique()]
    rows = []
    # Обычно месяцы идут строками, годы — колонками
    for i in range(raw.shape[0]):
        month_ts = parse_ru_month_year(raw.iat[i, 0])
        if pd.notna(month_ts):
            rows.append({"date": month_ts, "cpi_moscow_monthly": pd.to_numeric(raw.iat[i, 1], errors="coerce")})

    # fallback: вытягиваем из табличного блока месяц x год
    if not rows:
        month_rows = []
        for i in range(raw.shape[0]):
            month_name = str(raw.iat[i, 0]).strip().lower().replace("ё", "е")
            if month_name in RU_MONTHS:
                month_rows.append(i)
        year_cols = []
        for j in range(raw.shape[1]):
            cell = str(raw.iat[3, j]).strip()
            if re.fullmatch(r"20\d{2}", cell):
                year_cols.append((j, int(cell)))
        for i in month_rows:
            month = RU_MONTHS[str(raw.iat[i, 0]).strip().lower().replace("ё", "е")]
            for j, year in year_cols:
                value = pd.to_numeric(str(raw.iat[i, j]).replace(",", "."), errors="coerce")
                rows.append({"date": pd.Timestamp(year=year, month=month, day=1), "cpi_moscow_monthly": value})

    out = pd.DataFrame(rows).dropna(subset=["date"]).sort_values("date").drop_duplicates(subset=["date"])
    out = clip_date_range(out)
    q = aggregate_monthly_to_quarter(out, "cpi_moscow_monthly", "cpi_moscow_q", how="mean")
    return q


def parse_mosstat_housing_completion(url: str) -> pd.DataFrame:
    raw = pd.read_excel(url, header=None)
    rows = []
    # Ищем строки вида 'январь-март 2024' / 'I квартал 2024' / похожие
    for i in range(raw.shape[0]):
        blob = " ".join(map(str, raw.iloc[i].tolist())).lower().replace("ё", "е")
        qm = re.search(r"([ivx]+)\s*квартал\s*(20\d{2})", blob)
        if qm:
            roman = qm.group(1).upper()
            qmap = {"I":1, "II":2, "III":3, "IV":4}
            qnum = qmap.get(roman)
            year = int(qm.group(2))
            # Берём первое числовое значение в строке как тыс. кв. м
            vals = pd.to_numeric(pd.Series(raw.iloc[i].tolist()), errors="coerce").dropna()
            if not vals.empty:
                rows.append({"quarter": f"{year}Q{qnum}", "housing_completion_ths_sqm_q": vals.iloc[0]})

    # fallback: если в xlsx нет явных кварталов, но есть накопительные периоды январь-март / январь-июнь / ...
    if not rows:
        period_map = {
            "январь-март": 1,
            "январь-июнь": 2,
            "январь-сентябрь": 3,
            "январь-декабрь": 4,
        }
        for i in range(raw.shape[0]):
            blob = " ".join(map(str, raw.iloc[i].tolist())).lower().replace("ё", "е")
            for pat, qnum in period_map.items():
                m = re.search(pat + r"\s*(20\d{2})", blob)
                if m:
                    year = int(m.group(1))
                    vals = pd.to_numeric(pd.Series(raw.iloc[i].tolist()), errors="coerce").dropna()
                    if not vals.empty:
                        rows.append({"quarter": f"{year}Q{qnum}", "housing_completion_ths_sqm_cum": vals.iloc[0]})

        tmp = pd.DataFrame(rows).drop_duplicates(subset=["quarter"]).sort_values("quarter")
        if not tmp.empty and "housing_completion_ths_sqm_cum" in tmp.columns:
            out = tmp.copy()
            out["year"] = out["quarter"].str[:4]
            out["housing_completion_ths_sqm_q"] = out.groupby("year")["housing_completion_ths_sqm_cum"].diff()
            mask_q1 = out["quarter"].str.endswith("Q1")
            out.loc[mask_q1, "housing_completion_ths_sqm_q"] = out.loc[mask_q1, "housing_completion_ths_sqm_cum"]
            return out[["quarter", "housing_completion_ths_sqm_q"]].dropna(subset=["housing_completion_ths_sqm_q"])

    out = pd.DataFrame(rows).drop_duplicates(subset=["quarter"]).sort_values("quarter")
    if out.empty:
        raise RuntimeError("Не удалось распарсить ввод жилья из xlsx Мосстата")
    return out


In [ ]:

# Пытаемся найти и распарсить Мосстат.
# Если сайт/структура временно поменялись, ноутбук не падает целиком — просто пишет предупреждение.
mosstat_frames = {}

try:
    primary_xlsx = find_first_matching_xlsx(MOSSTAT_PAGES["primary_price_index"], MOSSTAT_PATTERNS["primary_price_index"])
    print("PRIMARY XLSX:", primary_xlsx)
    mosstat_frames["primary_price_index"] = parse_mosstat_primary_price_index(primary_xlsx)
    display(mosstat_frames["primary_price_index"].tail())
except Exception as e:
    print("[WARN] primary_price_index parser failed:", repr(e))

try:
    cpi_xlsx = find_first_matching_xlsx(MOSSTAT_PAGES["cpi"], MOSSTAT_PATTERNS["cpi"])
    print("CPI XLSX:", cpi_xlsx)
    mosstat_frames["cpi"] = parse_mosstat_cpi(cpi_xlsx)
    display(mosstat_frames["cpi"].tail())
except Exception as e:
    print("[WARN] cpi parser failed:", repr(e))

try:
    completion_xlsx = find_first_matching_xlsx(MOSSTAT_PAGES["housing_completion"], MOSSTAT_PATTERNS["housing_completion"])
    print("HOUSING COMPLETION XLSX:", completion_xlsx)
    mosstat_frames["housing_completion"] = parse_mosstat_housing_completion(completion_xlsx)
    display(mosstat_frames["housing_completion"].tail())
except Exception as e:
    print("[WARN] housing_completion parser failed:", repr(e))


## 6. Опциональные policy-флаги по льготной ипотеке

In [ ]:

# Здесь intentionally ручной календарь: события редкие, а сами флаги проще поддерживать вручную,
# чем парсить новостные страницы при каждом запуске.
# Основной количественный сигнал спроса всё равно лучше брать через фактические ипотечные объёмы/ставки.

policy = make_quarter_spine()
policy["quarter_start"] = pd.PeriodIndex(policy["quarter"], freq="Q").start_time
policy["quarter_end"]   = pd.PeriodIndex(policy["quarter"], freq="Q").end_time

# Массовая льготная ипотека ('Господдержка'): с 17.04.2020 по 01.07.2024
policy["subsidized_mortgage_2020_active_q"] = (
    (policy["quarter_end"] >= pd.Timestamp("2020-04-17")) &
    (policy["quarter_start"] < pd.Timestamp("2024-07-01"))
).astype(int)

# Новые правила семейной ипотеки / продление до 2030: используем как structural dummy с 2024Q3
policy["family_mortgage_new_rules_active_q"] = (policy["quarter_start"] >= pd.Timestamp("2024-07-01")).astype(int)

# Новые правила IT-ипотеки / продление до 2030: тоже structural dummy с 2024Q3
policy["it_mortgage_new_rules_active_q"] = (policy["quarter_start"] >= pd.Timestamp("2024-07-01")).astype(int)

policy = policy.drop(columns=["quarter_start", "quarter_end"])
display(policy.tail())


## 7. Склейка всех источников в единый quarterly CSV

In [ ]:
all_frames = [
    key_rate_q,
    usd_q,
    eur_q,
    mortgage_q,
    *moex_index_q_frames,
    policy,
]

for v in mosstat_frames.values():
    all_frames.append(v)

macro_q = outer_merge_on_quarter(all_frames).sort_values("quarter").reset_index(drop=True)

# Полезные derived-features
if {"key_rate_avg_q", "cpi_moscow_q"}.issubset(macro_q.columns):
    macro_q["real_policy_rate_q"] = macro_q["key_rate_avg_q"] - macro_q["cpi_moscow_q"]

if {"mortgage_total_volume_mln_rub_q", "mortgage_total_count_q"}.issubset(macro_q.columns):
    macro_q["mortgage_avg_ticket_mln_rub_q"] = macro_q["mortgage_total_volume_mln_rub_q"] / macro_q["mortgage_total_count_q"].replace(0, np.nan)

if {"mortgage_ddu_volume_mln_rub_q", "mortgage_ddu_count_q"}.issubset(macro_q.columns):
    macro_q["mortgage_ddu_avg_ticket_mln_rub_q"] = macro_q["mortgage_ddu_volume_mln_rub_q"] / macro_q["mortgage_ddu_count_q"].replace(0, np.nan)

# Сохраняем
macro_q.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print("Saved:", OUTPUT_CSV)
print("Shape:", macro_q.shape)
display(macro_q.tail(12))

coverage_df = build_coverage_table(macro_q)
coverage_df.to_csv(OUTPUT_COVERAGE_CSV, index=False, encoding="utf-8-sig")
print("Saved coverage:", OUTPUT_COVERAGE_CSV)
display(coverage_df)


## 8. Backfill Summary

Ниже выводится краткий markdown summary по новым ипотечным источникам, расширению coverage и оставшимся ограничениям.


In [ ]:
print("Head of macro dataset:")
display(macro_q.head(16))

print("Rows with key_rate present:")
display(macro_q.loc[macro_q["key_rate_avg_q"].notna(), ["quarter", "key_rate_avg_q", "key_rate_eoq"]].head(12))

print("Coverage summary for mortgage-related series:")
display(coverage_df.loc[coverage_df["feature"].str.contains("mortgage", na=False)].reset_index(drop=True))

mortgage_total_before = mortgage_coverage_compare.loc[mortgage_coverage_compare["feature"] == "mortgage_total_count_q", "first_quarter_before"].iloc[0]
mortgage_total_after = mortgage_coverage_compare.loc[mortgage_coverage_compare["feature"] == "mortgage_total_count_q", "first_quarter_after"].iloc[0]
mortgage_ddu_before = mortgage_coverage_compare.loc[mortgage_coverage_compare["feature"] == "mortgage_ddu_count_q", "first_quarter_before"].iloc[0]
mortgage_ddu_after = mortgage_coverage_compare.loc[mortgage_coverage_compare["feature"] == "mortgage_ddu_count_q", "first_quarter_after"].iloc[0]

retro_sources = mortgage_backfill_diag.get("sources", {})
pdf_diag = mortgage_backfill_diag.get("retro_pdf", {})
summary_lines = [
    "## Mortgage Backfill Summary",
    f"- Modern monthly layer: `{mortgage_current_diag.get('first_date')}` -> `{mortgage_current_diag.get('last_date')}` ({mortgage_current_diag.get('rows')} monthly rows).",
    f"- Retro XLSX layer: official CBR `retro2` Excel bundle `{retro_sources.get('retro_excel_url')}`.",
]
if retro_sources.get("stat_morgage_tables"):
    summary_lines.append(f"- Direct `stat_morgage_tables_XX.xlsx` probe found: `{retro_sources['stat_morgage_tables']}`.")
else:
    summary_lines.append("- Direct `stat_morgage_tables_XX.xlsx` probe did not return stable public URLs; notebook used the official `retro2` Excel bundle (`tableId=4-6`).")
summary_lines.extend([
    f"- Backfilled monthly mortgage block saved to `{OUTPUT_MORTGAGE_MONTHLY_CSV}`.",
    f"- `mortgage_total_*` quarterly history extended from `{mortgage_total_before}` to `{mortgage_total_after}`.",
    f"- `mortgage_ddu_*` quarterly history extended from `{mortgage_ddu_before}` to `{mortgage_ddu_after}`.",
])
if mortgage_backfill_diag.get("gap_metrics_after_retro_xlsx"):
    summary_lines.append(
        f"- Remaining early-gap metrics after retro XLSX: `{mortgage_backfill_diag['gap_metrics_after_retro_xlsx']}`."
    )
if pdf_diag.get("reason") == "no_pdf_tools":
    summary_lines.append("- PDF fallback was attempted conceptually, but skipped because `pdfplumber` / `camelot` / `tabula` are unavailable in this environment. Early DDU gaps before the first retro XLSX observation remain `NaN`.")
elif pdf_diag.get("used"):
    summary_lines.append("- PDF fallback contributed additional rows on top of retro XLSX.")
else:
    summary_lines.append("- Retro XLSX layer covered the needed period without invoking PDF parsing logic in practice.")

summary_lines.append(f"- Final quarterly CSV saved to `{OUTPUT_CSV}`, coverage CSV saved to `{OUTPUT_COVERAGE_CSV}`.")

display(Markdown("\n".join(summary_lines)))
